# PRE_05_data_formats

## Objetivo
Solución docente ejecutable del minicaso.

## Ejecución
Ejecute las celdas de arriba abajo desde este directorio.

## Solución
La celda siguiente materializa los datos y artefactos definidos por el PRE.

In [ ]:
"""Compara tres representaciones físicas del mismo conjunto lógico."""
import json
from pathlib import Path
import pandas as pd

ROOT = next(path for path in [Path.cwd().resolve(), *Path.cwd().resolve().parents] if (path / "data").is_dir() and (path / "submission").is_dir())
DATA, OUTPUT = ROOT / "data", ROOT / "submission/format_comparison.csv"


def build_submission():
    rows = [{"transaction_id": f"T{i:04d}", "transaction_date": f"2026-{(i % 3) + 1:02d}-{(i % 28) + 1:02d}", "customer_id": f"C{i % 40:03d}", "product_id": f"P{i % 20:03d}", "quantity": (i % 5) + 1, "unit_price": float((i % 9) + 2)} for i in range(1, 2001)]
    frame = pd.DataFrame(rows); frame["sales_amount"] = frame.quantity * frame.unit_price
    csv_path, json_path, parquet_path = DATA / "sales.csv", DATA / "sales.json", DATA / "sales.parquet"
    frame.to_csv(csv_path, index=False); frame.to_json(json_path, orient="records", date_format="iso"); frame.to_parquet(parquet_path, index=False, engine="pyarrow")
    assert frame.equals(pd.read_csv(csv_path).astype(frame.dtypes.to_dict()))
    assert frame.equals(pd.read_json(json_path).astype(frame.dtypes.to_dict()))
    assert frame.equals(pd.read_parquet(parquet_path, engine="pyarrow"))
    comparison = pd.DataFrame([
        ["CSV", csv_path.stat().st_size, False, True, False, "Intercambio e inspección simple"],
        ["JSON", json_path.stat().st_size, False, True, False, "Datos semiestructurados e intercambio"],
        ["Parquet", parquet_path.stat().st_size, True, False, True, "Lectura analítica columnar"],
    ], columns=["format","file_size_bytes","schema_preserved","human_readable","column_selection","primary_use_case"])
    comparison.to_csv(OUTPUT, index=False)


if __name__ == "__main__": build_submission()


## Verificación
Revise los artefactos generados en `submission/` y las pruebas automatizadas del PRE.